In [ ]:
import pandas as pd

recipes_df = pd.read_csv("./data/recipes.csv")

def gettransitionModel(recipes_df):
    result = {}

    for _, row in recipes_df.iterrows():
        meal_type = row["type"]
        meal_name = row["Name"]

        meal_info = (
            row["Category"],
            row["total_price"],
            row["provided_calories"],
            row["provided_protein"],
            row["provided_carbs"],
            row["provided_fat"]
        )

        if meal_type not in result:
            result[meal_type] = {}

        result[meal_type][meal_name] = meal_info

    return result

transition_model = gettransitionModel(recipes_df)

In [ ]:
class Node:
    def __init__(self, transition_model, slot_index,parent=None, meal_name=None):

        self.transition_model = transition_model
        self.slot_index = slot_index
        self.parent = parent
        self.meal_name = meal_name

        self.meal_types = ["Breakfast", "Lunch", "Dinner"]

        self.meal_type = self.meal_types[slot_index % 3] 
        self.price = 0
        self.calories = 0
        self.protein = 0
        self.carbs = 0
        self.fat = 0

        if self.meal_type and meal_name:
            info = transition_model[self.meal_type][meal_name]
            self.price = info[1]
            self.calories = info[2]
            self.protein = info[3]
            self.carbs = info[4]
            self.fat = info[5]

    #def is_goal(self):
    #return self.slot_index >= self.max_slots

    # def path(self):
    #     node = self
    #     result = []

    #     while node:
    #         if node.meal_name:
    #             result.append((node.meal_type, node.meal_name))
    #         node = node.parent

    #     return list(reversed(result))

In [ ]:
class MealPlanningProblem:

    def __init__(self, transition_model, TDEE, total_budget, num_days):
        self.transition_model = transition_model
        self.total_budget = total_budget
        self.TDEE = TDEE
        self.num_days = num_days
        self.total_slots = num_days * 3
        self.meal_types = ['Breakfast', 'Lunch', 'Dinner']

        self.meal_type_weights = {
            'Breakfast': 0.25,
            'Lunch': 0.40,
            'Dinner': 0.35,
        }

    def _path_total_price(self, node):
        total = 0
        current = node
        while current is not None:
            total += getattr(current, 'price', 0)
            current = current.parent
        return total

    def expand_node(self, node, use_cost=True, use_heuristic=False):
        if node.slot_index >= self.total_slots:
            return []

        children = []
        current_meal_type = self.meal_types[node.slot_index % 3]
        valid_actions = self.transition_model.get()

        current_meal_type = self.meal_types[node.slot_index % 3]
        candidates = self.transition_model.get(current_meal_type, {})

        base_cost = self._path_total_price(node)

        for meal_name, meal_info in candidates.items():
            meal_price = meal_info[1]
            new_total_price = base_cost + meal_price

            # Prune nodes that exceed the total budget.
            if new_total_price > self.total_budget:
                continue

            child = Node(
                transition_model=self.transition_model,
                slot_index=node.slot_index + 1,
                parent=node,
                meal_name=meal_name,
            )

            # Useful metadata for search algorithms (A*, Greedy, UCS...).
            child.action = meal_name
            child.path_cost = new_total_price if use_cost else 0

            if use_heuristic:
                # Heuristic: spending should roughly follow progress through slots.
                expected_spent = ((child.slot_index + 1) / self.total_slots) * self.total_budget
                child.heuristic = abs(new_total_price - expected_spent)
            else:
                child.heuristic = 0

            child.f = child.path_cost + child.heuristic
            children.append(child)

        return children